# اليوم الثاني — من البيانات الخام إلى أول نموذج
البيانات تُحمّل تلقائيًا من مستودع الدورة. شغّل الخلايا بالترتيب؛ جميع الأمثلة والقرارات محلولة ومشروحة.

## 0) تحميل بيانات منافذ تلقائيًا

In [ ]:
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم تحميل بيانات منافذ ✅')
print('عدد الصفوف والأعمدة:', df.shape)
display(df.head(3))

# المحطة 1 — افهم الجدول
ما الذي يمثله كل صف؟ ما عمود الهدف؟ اذكر ثلاث خصائص متاحة وقت التنبؤ.

In [ ]:
print(df.shape)
display(df.head())
display(df.dtypes.to_frame('نوع البيانات'))

# المحطة 2 — افحص الجودة ونظّف نسخة من البيانات

In [ ]:
df_clean = df.copy()
df_clean['signup_date'] = pd.to_datetime(df_clean['signup_date'], errors='coerce')
df_clean['snapshot_date'] = pd.to_datetime(df_clean['snapshot_date'], errors='coerce')
missing = df_clean.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('عدد القيم المفقودة'))
print('الصفوف المكررة:', df_clean.duplicated().sum())
display(df_clean['avg_rating'].describe())
display(df_clean.nlargest(5, 'avg_basket_sar')[['customer_id','avg_basket_sar','orders_per_month']])

## مثال محلول: قرار تنظيف القيمة المرتفعة
لا نحذف قيمة كبيرة تلقائيًا. المثال التالي يوضح قرارًا صحيحًا بعد فحص أعلى القيم.

In [ ]:
cleaning_decision = 'الإبقاء على القيمة وعدم حذفها'
reason = 'القيمة مرتفعة لكنها موجبة وممكنة منطقيًا، ولا يوجد دليل في البيانات أنها خطأ إدخال.'
print('القرار:', cleaning_decision)
print('السبب:', reason)

# المحطة 3 — جهّز X وy ومنع التسريب

In [ ]:
target = 'churned_30d'
drop_cols = ['customer_id', target, 'refund_issued', 'support_ticket_after_snapshot', 'next_month_orders', 'signup_date', 'snapshot_date']
X = df_clean.drop(columns=drop_cols, errors='ignore')
y = df_clean[target].copy()
forbidden = set(drop_cols) & set(X.columns)
assert not forbidden, f'تسريب: {forbidden}'
print('X:', X.shape, 'y:', y.shape)
print('لا توجد أعمدة ممنوعة داخل X ✅')

# المحطة 4 — تقسيم البيانات وبناء Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', num_pipe, numeric), ('cat', cat_pipe, categorical)])
print('التدريب:', X_train.shape, 'الاختبار:', X_test.shape)

# المحطة 5 — Baseline ثم نموذج تصنيف

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
models = {
 'Baseline': DummyClassifier(strategy='most_frequent'),
 'Logistic Regression': LogisticRegression(max_iter=1000),
 'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
}
rows = []
trained = {}
for name, estimator in models.items():
    model = Pipeline([('prep', clone(prep)), ('model', estimator)])
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    rows.append({'النموذج': name, 'Accuracy أولية': accuracy_score(y_test, prediction)})
    trained[name] = model
display(pd.DataFrame(rows).round(3))

## ما الذي شاهده الطالب؟
حجم البيانات، قرار تنظيف محلول، تجهيز X وy، إثبات منع التسريب، وجدول Baseline وLogistic Regression وDecision Tree. لا توجد إجابة ناقصة في هذا الدفتر.